# Explainable Hybrid Power Quality Disturbance Classification (Improved)
End-to-end implementation with fixes:
- Per-sample normalization (stabilizes training)
- Stronger 1D Residual TCN (BN + dropout + AdamW)
- Noise-augmented training (robust to SNR degradation)
- Residual DAE trained on random SNR (optional preprocessor)
- Clean 1D Grad-CAM (no Keras input-structure warning)


In [ ]:
import os, glob, math
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [ ]:
# =========================
# 1) Load XPQRS CSVs
# =========================
DATA_DIR = "/kaggle/input/seed-power-quality-disturbance-dataset/XPQRS"
csv_paths = glob.glob(os.path.join(DATA_DIR, "*.csv"))
assert len(csv_paths) > 0, f"No CSVs found in {DATA_DIR}. Update DATA_DIR."

dfs = []
for fp in csv_paths:
    label = os.path.splitext(os.path.basename(fp))[0]
    df0 = pd.read_csv(fp)
    df_long = df0.melt(var_name="instance", value_name="amplitude")
    df_long["time_idx"] = df_long.groupby("instance").cumcount()
    df_long["label"] = label
    dfs.append(df_long)

full_df = pd.concat(dfs, ignore_index=True)

pivot = full_df.pivot_table(index=["label","instance"], columns="time_idx", values="amplitude")
X_raw = pivot.values.astype("float32")  # (N, 999)
labels = pivot.index.get_level_values("label")

le = LabelEncoder()
y = le.fit_transform(labels)
num_classes = len(le.classes_)

print("Samples:", X_raw.shape[0], "Timesteps:", X_raw.shape[1], "Classes:", num_classes)
print("Classes:", list(le.classes_))


In [ ]:
# =========================
# 2) Train/Test split
# =========================
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=42
)

# =========================
# 3) Per-sample normalization (VERY IMPORTANT)
# Keeps shape info, removes scale bias
# =========================
def per_sample_standardize(x2d):
    mu = np.mean(x2d, axis=1, keepdims=True)
    sig = np.std(x2d, axis=1, keepdims=True) + 1e-8
    return (x2d - mu) / sig

X_train_raw_n = per_sample_standardize(X_train_raw)
X_test_raw_n  = per_sample_standardize(X_test_raw)

X_train_1d = X_train_raw_n[..., None].astype("float32")
X_test_1d  = X_test_raw_n[..., None].astype("float32")

print("Train:", X_train_1d.shape, "Test:", X_test_1d.shape)
print("Train mean/std:", float(X_train_1d.mean()), float(X_train_1d.std()))


## Phase 1: Residual Denoising Autoencoder (DAE)
This DAE is trained on **random SNR** (20–50 dB) and uses a **residual skip** to avoid over-smoothing. You can use it as an optional preprocessor for extreme noise conditions.

In [ ]:
# =========================
# Utility: AWGN noise by SNR (vectorized)
# =========================
def add_awgn_noise_snr_np(x, snr_db):
    # x: (N, T, 1)
    x = x.astype("float32")
    sig_pow = np.mean(x**2, axis=(1,2), keepdims=True)
    snr_lin = 10.0 ** (snr_db / 10.0)
    noise_pow = sig_pow / snr_lin
    noise = np.random.normal(0.0, 1.0, size=x.shape).astype("float32") * np.sqrt(noise_pow).astype("float32")
    return x + noise

# Build residual DAE
inp = layers.Input((999,1))

x = layers.Conv1D(32, 5, padding="same", activation="relu")(inp)
x = layers.MaxPool1D(2, padding="same")(x)
x = layers.Conv1D(16, 5, padding="same", activation="relu")(x)
x = layers.MaxPool1D(2, padding="same")(x)

x = layers.Conv1D(16, 5, padding="same", activation="relu")(x)
x = layers.UpSampling1D(2)(x)
x = layers.Conv1D(32, 5, padding="same", activation="relu")(x)
x = layers.UpSampling1D(2)(x)
x = layers.Conv1D(1, 5, padding="same")(x)
x = layers.Cropping1D((0,1))(x)

out = layers.Add()([inp, x])  # residual correction
autoencoder = Model(inp, out, name="Residual_DAE")
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()


In [ ]:
# =========================
# Train DAE on random SNR samples
# =========================
rng = np.random.default_rng(42)

def make_dae_training_pairs(X_clean, batch_size=32, snr_min=20, snr_max=50):
    N = X_clean.shape[0]
    idx = rng.permutation(N)
    for i in range(0, N, batch_size):
        b = idx[i:i+batch_size]
        clean = X_clean[b]
        snr = rng.uniform(snr_min, snr_max)
        noisy = add_awgn_noise_snr_np(clean, snr_db=snr)
        yield noisy, clean

# quick and simple training loop (works in Kaggle)
epochs_ae = 15
batch_size_ae = 32

for ep in range(1, epochs_ae+1):
    losses = []
    for noisy, clean in make_dae_training_pairs(X_train_1d, batch_size=batch_size_ae):
        loss = autoencoder.train_on_batch(noisy, clean)
        losses.append(loss)
    val_snr = 25
    val_noisy = add_awgn_noise_snr_np(X_test_1d, val_snr)
    val_loss = autoencoder.evaluate(val_noisy, X_test_1d, verbose=0)
    print(f"DAE Epoch {ep:02d}/{epochs_ae} | train_loss={float(np.mean(losses)):.4f} | val_loss@{val_snr}dB={float(val_loss):.4f}")


## Phase 2: Improved 1D Residual TCN
Changes vs previous version:
- BatchNorm + Dropout inside residual blocks
- Wider channels (64→128)
- AdamW optimizer
- Optional noise augmentation pipeline (recommended)

In [ ]:
def residual_tcn_block(x, filters, dilation_rate, dropout=0.2):
    h = layers.Conv1D(filters, 3, padding="causal", dilation_rate=dilation_rate)(x)
    h = layers.BatchNormalization()(h)
    h = layers.Activation("relu")(h)
    h = layers.Dropout(dropout)(h)

    h = layers.Conv1D(filters, 3, padding="causal", dilation_rate=dilation_rate)(h)
    h = layers.BatchNormalization()(h)

    if x.shape[-1] != filters:
        x = layers.Conv1D(filters, 1, padding="same")(x)

    h = layers.Add()([x, h])
    h = layers.Activation("relu")(h)
    return h

def build_1d_tcn(input_shape=(999,1), num_classes=17):
    inp = layers.Input(shape=input_shape)
    x = inp
    for d in [1,2,4,8,16,32,64]:
        x = residual_tcn_block(x, 64, d, dropout=0.2)

    x = layers.Conv1D(128, 3, padding="causal", activation="relu", name="target_conv_layer")(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return Model(inp, out, name="1D_Residual_TCN_Improved")

tcn_model = build_1d_tcn(input_shape=(999,1), num_classes=num_classes)

opt = tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4)
tcn_model.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
tcn_model.summary()


In [ ]:
# =========================
# Noise augmentation (recommended for robust SNR results)
# =========================
@tf.function
def augment_noise_tf(x, y):
    snr = tf.random.uniform([], 20.0, 50.0)  # dB
    xpow = tf.reduce_mean(tf.square(x))
    npow = xpow / (10.0 ** (snr / 10.0))
    noise = tf.random.normal(tf.shape(x), stddev=tf.sqrt(npow))
    return x + noise, y

batch_size = 32
train_ds = tf.data.Dataset.from_tensor_slices((X_train_1d, y_train))
train_ds = train_ds.shuffle(4096).batch(batch_size).map(augment_noise_tf, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_test_1d, y_test)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
]

history_tcn = tcn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=60,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
# =========================
# Clean evaluation on clean test
# =========================
preds = np.argmax(tcn_model.predict(X_test_1d, verbose=0), axis=1)
acc = accuracy_score(y_test, preds)
print("Clean Test Accuracy:", acc)

print("\nClassification report:")
print(classification_report(y_test, preds, target_names=le.classes_))


## Phase 3: Noise Immunity Profile (Fair)
We compare:
- Robust TCN (trained with noise augmentation)
- Robust TCN + Residual DAE preprocessor

If DAE is trained well, it should help mostly at **very low SNR**.

In [ ]:
snr_levels_db = [20, 25, 30, 35, 40, 45, 50]
tcn_acc = []
dae_tcn_acc = []

print("\n--- Running Noise Immunity Profile ---")
for snr in snr_levels_db:
    X_noisy = add_awgn_noise_snr_np(X_test_1d, snr)

    p1 = np.argmax(tcn_model.predict(X_noisy, verbose=0), axis=1)
    a1 = accuracy_score(y_test, p1)
    tcn_acc.append(a1)

    X_cleaned = autoencoder.predict(X_noisy, verbose=0)
    p2 = np.argmax(tcn_model.predict(X_cleaned, verbose=0), axis=1)
    a2 = accuracy_score(y_test, p2)
    dae_tcn_acc.append(a2)

    print(f"SNR {snr:>2} dB | TCN: {a1:.3f} | DAE+TCN: {a2:.3f}")

plt.figure(figsize=(10,6))
plt.plot(snr_levels_db, tcn_acc, marker="o", linestyle="--", label="Robust TCN")
plt.plot(snr_levels_db, dae_tcn_acc, marker="s", linestyle="-", label="DAE + Robust TCN")
plt.title("Noise Immunity Profile: Accuracy vs SNR")
plt.xlabel("SNR (dB)  [lower = noisier]")
plt.ylabel("Accuracy")
plt.grid(True, alpha=0.3)
plt.gca().invert_xaxis()
plt.legend()
plt.show()


## Phase 4: 1D Grad-CAM Explainability
Projects model attention onto the waveform. Warning fixed by passing `[inputs]` to the grad model.

In [ ]:
def make_gradcam_heatmap_1d(inputs, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        model.inputs,
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_out, preds = grad_model([inputs], training=False)  # <-- fixed structure warning
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_score = preds[:, pred_index]

    grads = tape.gradient(class_score, conv_out)          # (1, T, C)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1))      # (C,)

    conv_out = conv_out[0]                                # (T, C)
    heatmap = conv_out @ pooled_grads[..., tf.newaxis]    # (T, 1)
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def plot_gradcam_1d(signal, heatmap, title):
    t = np.linspace(0, 20, len(signal))  # 20ms cycle at 50Hz
    fig, ax = plt.subplots(figsize=(12,4))
    ax.plot(t, signal, linewidth=1.5, zorder=2)
    ymin, ymax = ax.get_ylim()
    extent = [t[0], t[-1], ymin, ymax]
    ax.imshow(heatmap[np.newaxis, :], aspect="auto", alpha=0.6, extent=extent, zorder=1)
    ax.set_title(title)
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Amplitude (normalized)")
    plt.tight_layout()
    plt.show()

sample_idx = 42
test_sample = X_test_1d[sample_idx:sample_idx+1]
true_label = le.inverse_transform([y_test[sample_idx]])[0]
pred_label = le.inverse_transform([int(np.argmax(tcn_model.predict(test_sample, verbose=0), axis=1)[0])])[0]

heat = make_gradcam_heatmap_1d(test_sample, tcn_model, "target_conv_layer")
plot_gradcam_1d(test_sample[0].squeeze(), heat, f"Grad-CAM | True: {true_label} | Pred: {pred_label}")
